# K-Means Clustering

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/clustering/01-k-means

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## K-Means Algorithm

1. Initialize centroids randomly
2. Assign each point to nearest centroid
3. Update centroids as mean of assigned points
4. Repeat until convergence

In [ ]:
np.random.seed(42)
K = 3
centers = np.array([[0, 0], [5, 0], [2.5, 4.3]])
n_per = 50
X = np.vstack([np.random.randn(n_per, 2) * 0.8 + c for c in centers])

def kmeans(X, K, n_iter=10):
    centroids = X[np.random.choice(len(X), K, replace=False)]
    history = [centroids.copy()]
    for _ in range(n_iter):
        dists = np.linalg.norm(X[:, None] - centroids[None], axis=2)
        labels = np.argmin(dists, axis=1)
        centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])
        history.append(centroids.copy())
    return labels, centroids, history

labels, centroids, history = kmeans(X, K)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    step_centroids = history[i]
    dists = np.linalg.norm(X[:, None] - step_centroids[None], axis=2)
    step_labels = np.argmin(dists, axis=1)
    for k in range(K):
        mask = step_labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=['#818cf8', '#14b8a6', '#eab308'][k], s=10, alpha=0.5)
    ax.scatter(step_centroids[:, 0], step_centroids[:, 1], c='white', s=200, marker='*', edgecolors='black', linewidths=1)
    ax.set_title(f'Iteration {i}', color='white', fontsize=11)
    ax.set_xlim(-2, 7)
    ax.set_ylim(-2, 6)
    ax.set_aspect('equal')
    ax.axis('off')
plt.suptitle('K-Means Convergence', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Why the update is the mean — and a worked example

With assignments fixed, cluster $k$ only appears in $\sum_{x_i \in C_k}\lVert x_i-\mu_k\rVert^2$. Setting the gradient to zero,

$$\frac{\partial}{\partial\mu_k}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert^2=\sum_{x_i\in C_k}-2(x_i-\mu_k)=0\;\Rightarrow\;\mu_k=\frac{1}{|C_k|}\sum_{x_i\in C_k}x_i.$$

So the **mean minimizes within-cluster squared distance** — that is *why* we move each centroid to its cluster mean. The assign step lowers $J$ (each point picks its nearest centroid) and the update step lowers $J$ (mean is optimal), so $J$ decreases monotonically. Below we verify both on the 5-point example from the lesson.

In [ ]:
import numpy as np

# 5-point worked example (A..E), K=2, init mu1=A, mu2=E
pts = {'A': (1, 1), 'B': (1.5, 2), 'C': (3, 4), 'D': (5, 7), 'E': (3.5, 5)}
P = np.array(list(pts.values())); names = list(pts.keys())
mu = np.array([[1.0, 1.0], [3.5, 5.0]])   # mu1 = A, mu2 = E

def sq(a, b):
    return float(np.sum((np.array(a) - np.array(b))**2))

def assign(P, mu):
    d = np.array([[sq(p, m) for m in mu] for p in P])
    return d, d.argmin(axis=1)

def inertia(P, labels, mu):
    return sum(sq(P[i], mu[labels[i]]) for i in range(len(P)))

# --- Iteration 1: assign (squared-distance table) ---
d, labels = assign(P, mu)
print('Iter 1 assign  | d^2 to mu1   d^2 to mu2  -> cluster')
for i, n in enumerate(names):
    print(f'  {n}{tuple(P[i])}  |   {d[i,0]:6.2f}      {d[i,1]:6.2f}    ->  C{labels[i]+1}')
J_before = inertia(P, labels, mu)
print(f'\nJ with OLD centroids (post-assign) = {J_before:.2f}')

# --- Iteration 1: update (mean of each cluster) ---
mu_new = np.array([P[labels == k].mean(axis=0) for k in range(2)])
J_after = inertia(P, labels, mu_new)
print(f'New centroids: mu1={tuple(np.round(mu_new[0],3))}, mu2={tuple(np.round(mu_new[1],3))}')
print(f'J with NEW centroids               = {J_after:.2f}   ({J_after:.2f} < {J_before:.2f}  ✓ update lowered J)')

# Verify the mean BEATS any nearby centroid choice (mean is the minimizer)
c0 = P[labels == 1]                      # cluster C2 points
grid = c0.mean(0) + np.random.default_rng(0).normal(0, 0.3, size=(2000, 2))
best = min(sum(sq(p, g) for p in c0) for g in grid)
mean_sse = sum(sq(p, c0.mean(0)) for p in c0)
print(f'\nMean SSE for C2 = {mean_sse:.3f};  best of 2000 random nearby centroids = {best:.3f}'
      f'  -> mean is optimal: {mean_sse <= best}')

# --- Iteration 2: reassign with new centroids -> should be unchanged (converged) ---
_, labels2 = assign(P, mu_new)
print(f'\nIter 2 labels {list(labels2)}  ==  Iter 1 labels {list(labels)}  -> converged: {np.array_equal(labels, labels2)}')
print('Final: C1 =', [names[i] for i in range(5) if labels[i]==0],
      ' C2 =', [names[i] for i in range(5) if labels[i]==1])


## Elbow Method

Plot inertia vs K — look for the "elbow" where improvement slows.

In [ ]:
inertias = []
for k in range(1, 10):
    _, cents, _ = kmeans(X, k)
    dists = np.linalg.norm(X[:, None] - cents[None], axis=2)
    labels = np.argmin(dists, axis=1)
    inertia = sum(np.sum((X[labels == k] - cents[k])**2) for k in range(k))
    inertias.append(inertia)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, 10), inertias, 'o-', color='#818cf8', linewidth=2, markersize=8)
axes[0].axvline(3, color='#f43f5e', linestyle='--', alpha=0.7, label='Elbow at K=3')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia (Within-Cluster SS)')
axes[0].set_title('Elbow Method', color='white')
axes[0].legend()

silhouette_scores = []
for k in range(2, 10):
    _, cents, _ = kmeans(X, k)
    dists = np.linalg.norm(X[:, None] - cents[None], axis=2)
    labels = np.argmin(dists, axis=1)
    s = 0
    for i in range(len(X)):
        ci = labels[i]
        a = np.mean(np.linalg.norm(X[labels == ci] - X[i], axis=1))
        b = min(np.mean(np.linalg.norm(X[labels == j] - X[i], axis=1)) for j in range(k) if j != ci)
        s += (b - a) / max(a, b)
    silhouette_scores.append(s / len(X))
axes[1].plot(range(2, 10), silhouette_scores, 's-', color='#14b8a6', linewidth=2, markersize=8)
axes[1].axvline(3, color='#f43f5e', linestyle='--', alpha=0.7, label='Best at K=3')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', color='white')
axes[1].legend()
plt.tight_layout()
plt.show()

## Silhouette score: validating K

The silhouette compares each point's cohesion (own cluster) to its separation (nearest other cluster). Values near 1 are good; the best $K$ maximizes the mean.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=0)
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f'K={k}: silhouette = {silhouette_score(X, labels):.3f}')

## Key takeaways

- K-Means alternates **assign to nearest centroid** and **move centroid to mean** until stable.
- It minimizes within-cluster variance (inertia) but assumes spherical, similar-size clusters.
- Pick $K$ with the **elbow** (inertia) or **silhouette** score; use **K-Means++** init and `n_init>1`.
- Always **standardize** features — K-Means uses Euclidean distance.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The assign step

Half of K-Means: send every point to its **nearest centroid**. Implement it vectorized — compute all point-to-centroid distances, then `argmin` across centroids.

In [ ]:
def assign_clusters(X, centroids):
    """Label each point with the index of its nearest centroid."""
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(centroids, dtype=float)

    # TODO(you): distances of every point to every centroid
    # (hint: np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2))
    dists = ...

    # TODO(you): index of the nearest centroid per point (hint: np.argmin with axis=1)
    return ...

In [ ]:
# Checks — run me
X = np.array([[0.0, 0.0], [1.0, 0.0], [10.0, 10.0], [11.0, 10.0]])
C = np.array([[0.5, 0.0], [10.5, 10.0]])
assert list(assign_clusters(X, C)) == [0, 0, 1, 1], "each point joins its nearest centroid"
assert list(assign_clusters([[4.9, 0.0]], np.array([[0.0, 0.0], [10.0, 0.0]]))) == [0], \
    "4.9 is closer to 0 than to 10"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def assign_clusters(X, centroids):
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(centroids, dtype=float)
    dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
    return np.argmin(dists, axis=1)
```

</details>

### Exercise 2 — The update step (and convergence)

The other half: each centroid moves to the **mean** of the points assigned to it — the mean is exactly the point that minimizes within-cluster squared distance, which is why K-Means converges. The second check catches the moment it does: when assigning and updating leave the centroids unchanged, you've hit a fixed point.

In [ ]:
def update_centroids(X, labels, k):
    """New centroid j = mean of the points with label j."""
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)

    # TODO(you): per-cluster means, stacked into a (k, d) array
    return ...

In [ ]:
# Checks — run me
X = np.array([[0.0, 0.0], [1.0, 0.0], [10.0, 10.0], [11.0, 10.0]])
new_C = update_centroids(X, np.array([0, 0, 1, 1]), 2)
assert np.allclose(new_C, [[0.5, 0.0], [10.5, 10.0]]), "each centroid moves to its cluster's mean"

labels = assign_clusters(X, new_C)
assert np.allclose(update_centroids(X, labels, 2), new_C), \
    "assign + update changes nothing: K-Means has converged"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def update_centroids(X, labels, k):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    return np.array([X[labels == j].mean(axis=0) for j in range(k)])
```

</details>